# BRFSS 2015 - heterogeneous ensemble (weighted soft voting + OOF stacking)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laiba-Gul/explainable-diabetes-prediction/blob/main/notebooks/07_brfss_heterogeneous_ensemble.ipynb)

**Status:** supporting research notebook (original Colab work, cleaned). Outputs were cleared to keep the repository small and free of personal metadata; re-run it or see the reproducible results in [`results/`](../results/).

**What it does:** Five tree learners at natural prevalence, validation-optimised soft-voting weights, memory-safe out-of-fold stacking, validation-only threshold selection, permutation importance, SHAP and LIME. Reproduced by `configs/brfss_ensemble.yaml`.

**Data:** run `diabetes-xai download` (or upload the CSV when the notebook asks). The CSVs are checksum-verified and are not stored in Git.


# Early Diabetes Prediction on the Original BRFSS Dataset

**Proposed model:** leakage-safe heterogeneous ensemble combining optimized weighted soft voting and out-of-fold stacking  
**User-supplied dataset:** `diabetes_binary_health_indicators_BRFSS2015.csv`  
**Rows:** 253,680 | **Predictors:** 21 | **Positive class:** approximately 13.93%

This notebook uses the **first of the two supplied BRFSS datasets**, not an external dataset. It tests whether 95% accuracy and positive-class precision can be reached honestly. It does not insert a target score, search seeds, balance before splitting, tune on test labels, or report training performance as final performance.

Two validation-selected operating points are frozen before the test gate:

1. **Accuracy mode:** maximizes validation accuracy.
2. **High-precision mode:** maximizes diabetic-case recall while targeting at least 95% precision and at least 10% recall.

If the dataset cannot support 95%, the notebook reports the actual result. A missed target must not be repaired by reusing the same test set.

## 0. Google Colab setup

In Colab, select **Runtime → Change runtime type → T4 GPU** when available. Upload the original CSV when prompted. A full run can take 20–45 minutes depending on the runtime.

In [ ]:
%pip -q install -U xgboost lightgbm catboost shap lime

In [ ]:
import gc, json, math, os, platform, random, shutil, sys, time, warnings
from collections import OrderedDict
from pathlib import Path

# Keep native numerical libraries from creating oversized thread pools in Colab.
for thread_variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[thread_variable] = "2"

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from scipy.optimize import minimize
from scipy.stats import entropy
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, log_loss,
    matthews_corrcoef, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
SEED = 42
CPU_THREADS = max(1, min(2, os.cpu_count() or 1))
random.seed(SEED)
np.random.seed(SEED)

RESULTS_DIR = Path("brfss_ensemble_high_performance_results")
FIG_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
MODEL_DIR = RESULTS_DIR / "models"
for folder in (RESULTS_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
SOFT = ["#A8DADC", "#FFB4A2", "#CDB4DB", "#BDE0FE", "#F6BD60", "#84A59D"]

TARGET_ACCURACY = 0.95
TARGET_PRECISION = 0.95
MIN_HIGH_PRECISION_RECALL = 0.10
print(f"Environment ready. The random seed is fixed and CPU threads are capped at {CPU_THREADS} to protect Colab RAM.")

## 1. Load and audit the supplied original BRFSS dataset

The notebook accepts only the original 253,680-row file and validates its schema, target, row count, and class distribution before modeling.

In [ ]:
EXPECTED_FILE = "diabetes_binary_health_indicators_BRFSS2015.csv"
DATA_PATH = Path("/content") / EXPECTED_FILE

candidates = [
    DATA_PATH,
    Path(EXPECTED_FILE),
    Path("/content/drive/MyDrive") / EXPECTED_FILE,
]
existing = next((p for p in candidates if p.exists()), None)

if existing is None:
    print(f"Upload {EXPECTED_FILE}")
    from google.colab import files
    uploaded = files.upload()
    if EXPECTED_FILE in uploaded:
        existing = Path(EXPECTED_FILE)
    elif len(uploaded) == 1:
        existing = Path(next(iter(uploaded)))
    else:
        raise RuntimeError(f"Please upload exactly one file: {EXPECTED_FILE}")

df = pd.read_csv(existing)
TARGET = "Diabetes_binary"
EXPECTED_FEATURES = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income",
]
expected_columns = [TARGET] + EXPECTED_FEATURES
missing = sorted(set(expected_columns) - set(df.columns))
extra = sorted(set(df.columns) - set(expected_columns))
if missing or extra:
    raise ValueError(f"Schema mismatch. Missing={missing}; extra={extra}")

df = df[expected_columns].copy()
df[TARGET] = pd.to_numeric(df[TARGET], errors="raise").astype(int)
if set(df[TARGET].unique()) != {0, 1}:
    raise ValueError(f"Target must contain only 0 and 1; found {sorted(df[TARGET].unique())}")
for column in EXPECTED_FEATURES:
    df[column] = pd.to_numeric(df[column], errors="coerce")

audit = {
    "source_file": existing.name,
    "rows": int(len(df)),
    "predictors": len(EXPECTED_FEATURES),
    "negative_n": int((df[TARGET] == 0).sum()),
    "positive_n": int((df[TARGET] == 1).sum()),
    "positive_percent": float(100 * df[TARGET].mean()),
    "missing_cells": int(df.isna().sum().sum()),
    "exact_duplicate_rows": int(df.duplicated(keep=False).sum()),
}
if len(df) != 253_680:
    raise ValueError(f"Expected 253,680 rows for the original dataset; found {len(df):,}")

(RESULTS_DIR / "dataset_audit.json").write_text(json.dumps(audit, indent=2))
print(json.dumps(audit, indent=2))
display(df.head())

In [ ]:
counts = df[TARGET].value_counts().reindex([0, 1], fill_value=0)
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    counts.values,
    labels=["Non-diabetic", "Diabetic"],
    autopct="%1.2f%%",
    startangle=90,
    colors=SOFT[:2],
    wedgeprops={"edgecolor": "white", "linewidth": 2},
)
ax.set_title("Original BRFSS class distribution")
plt.tight_layout()
plt.savefig(FIG_DIR / "class_distribution_soft_pie.png", dpi=300, bbox_inches="tight")
plt.show()

## 2. Stratified 70% / 15% / 15% split

The split is performed before any learned preprocessing, feature analysis, model fitting, threshold selection, or class treatment. The test labels are stored behind a final gate and are not used for model or threshold selection.

Repeated BRFSS response profiles are retained because this public file has no respondent identifier and identical answers do not prove that records belong to the same person. This limitation must be stated in the paper.

In [ ]:
X = df[EXPECTED_FEATURES].copy()
y = df[TARGET].copy()
row_ids = np.arange(len(df))

X_train, X_temp, y_train, y_temp, id_train, id_temp = train_test_split(
    X, y, row_ids,
    test_size=0.30,
    stratify=y,
    random_state=SEED,
)
X_val, X_test_locked, y_val, _y_test_locked, id_val, id_test = train_test_split(
    X_temp, y_temp, id_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=SEED,
)

X_train, y_train = X_train.reset_index(drop=True), y_train.reset_index(drop=True)
X_val, y_val = X_val.reset_index(drop=True), y_val.reset_index(drop=True)
X_test_locked = X_test_locked.reset_index(drop=True)
_y_test_locked = _y_test_locked.reset_index(drop=True)

split_rows = []
for name, yy in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Testing (locked)", _y_test_locked),
]:
    split_rows.append({
        "split": name,
        "n": int(len(yy)),
        "percent_of_total": 100 * len(yy) / len(y),
        "negative_n": int((yy == 0).sum()),
        "positive_n": int((yy == 1).sum()),
        "positive_percent": 100 * yy.mean(),
    })
split_summary = pd.DataFrame(split_rows)
split_summary.to_csv(TABLE_DIR / "split_summary.csv", index=False)
display(split_summary.style.format({"percent_of_total": "{:.2f}", "positive_percent": "{:.2f}"}))

assert len(X_train) == 177_576 and len(X_val) == 38_052 and len(X_test_locked) == 38_052
TEST_SET_LOCKED = True
del df, X, y, row_ids, X_temp, y_temp, id_train, id_val
gc.collect()
print("TEST SET LOCKED: 38,052 rows.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, yy) in zip(
    axes,
    [("Training", y_train), ("Validation", y_val), ("Testing", _y_test_locked)],
):
    cc = yy.value_counts().reindex([0, 1], fill_value=0)
    ax.pie(
        cc.values,
        labels=["Non-diabetic", "Diabetic"],
        autopct="%1.2f%%",
        startangle=90,
        colors=SOFT[:2],
        wedgeprops={"edgecolor": "white"},
    )
    ax.set_title(name)
fig.suptitle("Stratified split class distributions")
plt.tight_layout()
plt.savefig(FIG_DIR / "split_distribution_soft_pies.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Training-only feature-selection analysis

Correlation, information gain/mutual information, gain ratio, and Extra Trees importance are computed from the training split only. All 21 predictors are retained for the proposed ensemble because the feature space is small and the goal is maximum honest predictive performance. The table remains useful for interpretation and later ablation studies.

In [ ]:
X_train_analysis = X_train.copy()
train_medians = X_train_analysis.median()
X_train_analysis = X_train_analysis.fillna(train_medians)

mutual_information = mutual_info_classif(
    X_train_analysis,
    y_train,
    discrete_features=True,
    random_state=SEED,
)
correlations = X_train_analysis.apply(lambda col: col.corr(y_train))

predictor_entropies = []
for feature in EXPECTED_FEATURES:
    probabilities = X_train_analysis[feature].value_counts(normalize=True).to_numpy()
    predictor_entropies.append(float(entropy(probabilities, base=2)))
predictor_entropies = np.asarray(predictor_entropies)
gain_ratio = np.divide(
    mutual_information,
    predictor_entropies,
    out=np.zeros_like(mutual_information),
    where=predictor_entropies > 0,
)

feature_probe = ExtraTreesClassifier(
    n_estimators=400,
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=CPU_THREADS,
)
feature_probe.fit(X_train_analysis, y_train)

feature_analysis = pd.DataFrame({
    "feature": EXPECTED_FEATURES,
    "correlation_with_target": correlations.values,
    "information_gain_mutual_information": mutual_information,
    "gain_ratio": gain_ratio,
    "extra_trees_importance": feature_probe.feature_importances_,
}).sort_values("information_gain_mutual_information", ascending=False)
feature_analysis.to_csv(TABLE_DIR / "training_only_feature_selection_analysis.csv", index=False)
display(feature_analysis)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
plots = [
    ("correlation_with_target", "Correlation with target", SOFT[0]),
    ("information_gain_mutual_information", "Information gain / mutual information", SOFT[1]),
    ("gain_ratio", "Gain ratio", SOFT[2]),
    ("extra_trees_importance", "Extra Trees importance", SOFT[3]),
]
for ax, (column, title, color) in zip(axes.flat, plots):
    data = feature_analysis.sort_values(column)
    ax.barh(data["feature"], data[column], color=color)
    ax.set_title(title)
plt.tight_layout()
plt.savefig(FIG_DIR / "training_only_feature_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

# Release the 400-tree feature probe before fitting the much larger ensembles.
del feature_probe, X_train_analysis
gc.collect()
print("Feature-analysis working objects released from RAM.")

## 4. Class-balance decision

The original dataset is imbalanced, but the proposed primary model does **not** use SMOTE because the user requested maximum accuracy and precision and synthetic oversampling can reduce calibration and overall accuracy. The natural training prevalence is preserved. If a later sensitivity analysis adds SMOTE, it must be fitted inside training folds only; validation and test distributions must remain unchanged.

In [ ]:
balance_decision = {
    "training_negative_n": int((y_train == 0).sum()),
    "training_positive_n": int((y_train == 1).sum()),
    "training_positive_percent": float(100 * y_train.mean()),
    "smote_applied": False,
    "reason": "Natural prevalence retained for calibrated high-accuracy/high-precision evaluation.",
    "validation_resampled": False,
    "test_resampled": False,
}
(RESULTS_DIR / "balance_decision.json").write_text(json.dumps(balance_decision, indent=2))
print(json.dumps(balance_decision, indent=2))

## 5. Train heterogeneous base learners

The ensemble combines five diverse tree-based learners: Extra Trees, Histogram Gradient Boosting, XGBoost, LightGBM, and CatBoost. Missing-value handling is learned from training data only. No model sees the test labels during fitting or selection.

In [ ]:
train_medians = X_train.median()
X_train_fit = X_train.fillna(train_medians)
X_val_fit = X_val.fillna(train_medians)
X_test_fit_locked = X_test_locked.fillna(train_medians)

base_models = OrderedDict({
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=700,
        max_features=0.80,
        min_samples_leaf=2,
        max_depth=None,
        random_state=SEED,
        n_jobs=CPU_THREADS,
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        learning_rate=0.06,
        max_iter=400,
        max_leaf_nodes=23,
        min_samples_leaf=30,
        l2_regularization=2.0,
        early_stopping=True,
        random_state=SEED,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=900,
        learning_rate=0.035,
        max_depth=5,
        min_child_weight=8,
        subsample=0.85,
        colsample_bytree=0.90,
        reg_alpha=0.05,
        reg_lambda=3.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=SEED,
        n_jobs=CPU_THREADS,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=40,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=2.0,
        objective="binary",
        random_state=SEED,
        n_jobs=CPU_THREADS,
        verbosity=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=900,
        learning_rate=0.04,
        depth=7,
        l2_leaf_reg=5.0,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
        thread_count=CPU_THREADS,
    ),
})

fit_rows = []
validation_base_probabilities = OrderedDict()
for model_name, model in base_models.items():
    start = time.time()
    print(f"Training {model_name}...")
    model.fit(X_train_fit, y_train)
    probabilities = model.predict_proba(X_val_fit)[:, 1]
    validation_base_probabilities[model_name] = probabilities
    fit_rows.append({
        "model": model_name,
        "fit_seconds": time.time() - start,
        "validation_roc_auc": roc_auc_score(y_val, probabilities),
        "validation_log_loss": log_loss(y_val, np.clip(probabilities, 1e-7, 1-1e-7)),
    })

base_validation_summary = pd.DataFrame(fit_rows).sort_values("validation_roc_auc", ascending=False)
base_validation_summary.to_csv(TABLE_DIR / "base_model_validation_summary.csv", index=False)
display(base_validation_summary.style.format({
    "fit_seconds": "{:.1f}", "validation_roc_auc": "{:.4f}", "validation_log_loss": "{:.4f}"
}))

## 6. Build two new ensemble candidates

1. **Optimized weighted soft voting:** non-negative weights summing to one are selected by validation log loss.
2. **Memory-safe out-of-fold stacking:** Extra Trees, HistGradientBoosting, and LightGBM feed a logistic meta-learner using three-fold out-of-fold training predictions. Folds run sequentially, with one temporary model held in RAM at a time.

Only these two new ensembles are eligible as the final proposed model; the five individual models are components and benchmarks.

In [ ]:
base_names = list(validation_base_probabilities)
val_matrix = np.column_stack([validation_base_probabilities[name] for name in base_names])

def softmax(raw):
    shifted = raw - np.max(raw)
    values = np.exp(shifted)
    return values / values.sum()

def blend_loss(raw):
    weights = softmax(raw)
    probabilities = np.clip(val_matrix @ weights, 1e-7, 1 - 1e-7)
    return log_loss(y_val, probabilities)

blend_result = minimize(
    blend_loss,
    x0=np.zeros(len(base_names), dtype=float),
    method="L-BFGS-B",
)
blend_weights = softmax(blend_result.x)
weighted_val_probabilities = val_matrix @ blend_weights
weight_table = pd.DataFrame({"model": base_names, "weight": blend_weights})
weight_table.to_csv(TABLE_DIR / "optimized_soft_voting_weights.csv", index=False)
display(weight_table.style.format({"weight": "{:.4f}"}))

# Generate genuine OOF predictions sequentially. This avoids joblib worker
# processes duplicating large tree ensembles and exhausting Colab RAM.
STACK_BASE_NAMES = ["ExtraTrees", "HistGradientBoosting", "LightGBM"]
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
X_train_stack = X_train_fit.astype(np.float32)
oof_stack_matrix = np.empty((len(X_train_stack), len(STACK_BASE_NAMES)), dtype=np.float32)
stack_fold_rows = []

print("Training the memory-safe out-of-fold stacking ensemble...")
for fold_number, (fold_fit_idx, fold_oof_idx) in enumerate(inner_cv.split(X_train_stack, y_train), start=1):
    X_fold_fit = X_train_stack.iloc[fold_fit_idx]
    y_fold_fit = y_train.iloc[fold_fit_idx]
    X_fold_oof = X_train_stack.iloc[fold_oof_idx]
    for model_index, model_name in enumerate(STACK_BASE_NAMES):
        print(f"  Fold {fold_number}/3 - {model_name}")
        fold_start = time.time()
        fold_model = clone(base_models[model_name])
        # One model and one native thread pool at a time: no nested workers.
        if model_name in {"ExtraTrees", "LightGBM"}:
            fold_model.set_params(n_jobs=1)
        fold_model.fit(X_fold_fit, y_fold_fit)
        oof_stack_matrix[fold_oof_idx, model_index] = (
            fold_model.predict_proba(X_fold_oof)[:, 1].astype(np.float32)
        )
        stack_fold_rows.append({
            "fold": fold_number,
            "model": model_name,
            "fit_rows": int(len(fold_fit_idx)),
            "oof_rows": int(len(fold_oof_idx)),
            "seconds": time.time() - fold_start,
        })
        del fold_model
        gc.collect()
    del X_fold_fit, y_fold_fit, X_fold_oof
    gc.collect()

pd.DataFrame(stack_fold_rows).to_csv(TABLE_DIR / "stacking_fold_training_log.csv", index=False)
stack_meta_features = np.column_stack([
    oof_stack_matrix,
    X_train_stack.to_numpy(dtype=np.float32, copy=False),
]).astype(np.float32, copy=False)
stack_meta_model = Pipeline([
    ("scaler", StandardScaler()),
    ("meta", LogisticRegression(C=0.5, max_iter=3000, random_state=SEED)),
])
stack_meta_model.fit(stack_meta_features, y_train)

def stack_predict_proba(frame):
    stack_base_matrix = np.column_stack([
        base_models[name].predict_proba(frame)[:, 1]
        for name in STACK_BASE_NAMES
    ]).astype(np.float32)
    stack_meta_frame = np.column_stack([
        stack_base_matrix,
        frame.to_numpy(dtype=np.float32, copy=False),
    ]).astype(np.float32, copy=False)
    return stack_meta_model.predict_proba(stack_meta_frame)

del stack_meta_features, oof_stack_matrix, X_train_stack
gc.collect()
stacking_val_probabilities = stack_predict_proba(X_val_fit)[:, 1]

validation_ensemble_probabilities = OrderedDict({
    "OptimizedWeightedSoftVoting": weighted_val_probabilities,
    "OutOfFoldStacking": stacking_val_probabilities,
})
print("Both new ensemble candidates are ready. The test set remains locked.")

## 7. Validation-only model and threshold selection

The final ensemble and two thresholds are selected using validation labels only. The high-precision threshold cannot qualify by predicting only a handful of positives: it must retain at least 10% diabetic recall on validation data.

In [ ]:
METRIC_COLUMNS = [
    "accuracy", "precision", "recall", "f1", "roc_auc",
    "mcc", "specificity", "sensitivity", "log_loss",
]

def metric_bundle(y_true, probabilities, threshold):
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    sensitivity = recall_score(y_true, predictions, zero_division=0)
    return {
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": sensitivity,
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "mcc": matthews_corrcoef(y_true, predictions),
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "sensitivity": sensitivity,
        "log_loss": log_loss(y_true, np.clip(probabilities, 1e-7, 1 - 1e-7), labels=[0, 1]),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def threshold_search(y_true, probabilities):
    rows = []
    for threshold in np.linspace(0.01, 0.99, 981):
        row = metric_bundle(y_true, probabilities, threshold)
        row["threshold"] = float(threshold)
        row["dual_95_target"] = bool(
            row["accuracy"] >= TARGET_ACCURACY
            and row["precision"] >= TARGET_PRECISION
            and row["recall"] >= MIN_HIGH_PRECISION_RECALL
        )
        rows.append(row)
    table = pd.DataFrame(rows)

    accuracy_choice = table.sort_values(
        ["accuracy", "mcc", "precision", "f1"], ascending=False
    ).iloc[0]

    precision_pool = table[table["recall"] >= MIN_HIGH_PRECISION_RECALL]
    precision_feasible = precision_pool[precision_pool["precision"] >= TARGET_PRECISION]
    if len(precision_feasible):
        precision_choice = precision_feasible.sort_values(
            ["recall", "accuracy", "mcc"], ascending=False
        ).iloc[0]
        precision_mode = "precision_target_met_on_validation"
    else:
        precision_choice = precision_pool.sort_values(
            ["precision", "recall", "accuracy"], ascending=False
        ).iloc[0]
        precision_mode = "best_precision_with_minimum_recall"
    return accuracy_choice, precision_choice, precision_mode, table

selection_rows = []
selection_objects = {}
for model_name, probabilities in validation_ensemble_probabilities.items():
    acc_choice, prec_choice, prec_mode, table = threshold_search(y_val, probabilities)
    table.to_csv(TABLE_DIR / f"{model_name}_validation_threshold_search.csv", index=False)
    selection_objects[model_name] = {
        "accuracy": acc_choice,
        "precision": prec_choice,
        "precision_mode": prec_mode,
    }
    selection_rows.append({
        "model": model_name,
        "accuracy_threshold": float(acc_choice["threshold"]),
        "accuracy_mode_accuracy": float(acc_choice["accuracy"]),
        "accuracy_mode_precision": float(acc_choice["precision"]),
        "accuracy_mode_recall": float(acc_choice["recall"]),
        "precision_threshold": float(prec_choice["threshold"]),
        "precision_mode_accuracy": float(prec_choice["accuracy"]),
        "precision_mode_precision": float(prec_choice["precision"]),
        "precision_mode_recall": float(prec_choice["recall"]),
        "precision_selection_mode": prec_mode,
        "validation_roc_auc": float(acc_choice["roc_auc"]),
        "dual_95_target_on_validation": bool(table["dual_95_target"].any()),
    })

selection_table = pd.DataFrame(selection_rows).sort_values(
    ["accuracy_mode_accuracy", "validation_roc_auc", "accuracy_mode_precision"],
    ascending=False,
).reset_index(drop=True)
selection_table.to_csv(TABLE_DIR / "validation_ensemble_selection.csv", index=False)
display(selection_table.style.format({c: "{:.4f}" for c in selection_table.select_dtypes("number").columns}))

FINAL_MODEL_NAME = selection_table.loc[0, "model"]
FINAL_ACCURACY_THRESHOLD = float(selection_table.loc[0, "accuracy_threshold"])
FINAL_PRECISION_THRESHOLD = float(selection_table.loc[0, "precision_threshold"])
MODEL_AND_THRESHOLDS_FROZEN = True

frozen = {
    "final_model": FINAL_MODEL_NAME,
    "accuracy_threshold": FINAL_ACCURACY_THRESHOLD,
    "high_precision_threshold": FINAL_PRECISION_THRESHOLD,
    "selection_source": "validation_only",
    "random_seed": SEED,
    "test_used_for_selection": False,
    "target_accuracy": TARGET_ACCURACY,
    "target_precision": TARGET_PRECISION,
    "minimum_high_precision_recall": MIN_HIGH_PRECISION_RECALL,
}
(RESULTS_DIR / "frozen_model_and_thresholds.json").write_text(json.dumps(frozen, indent=2))
print(json.dumps(frozen, indent=2))
print("MODEL AND BOTH THRESHOLDS FROZEN. No later adjustment is permitted.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for model_name, probabilities in validation_base_probabilities.items():
    fpr, tpr, _ = roc_curve(y_val, probabilities)
    ax.plot(fpr, tpr, alpha=0.45, linewidth=1.2, label=f"{model_name} ({roc_auc_score(y_val, probabilities):.3f})")
for model_name, probabilities in validation_ensemble_probabilities.items():
    fpr, tpr, _ = roc_curve(y_val, probabilities)
    ax.plot(fpr, tpr, linewidth=2.8, label=f"{model_name} ({roc_auc_score(y_val, probabilities):.3f})")
ax.plot([0, 1], [0, 1], "--", color="#777777")
ax.set(xlabel="False-positive rate", ylabel="True-positive rate", title="Validation ROC: base learners and new ensembles")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "validation_combined_roc.png", dpi=300, bbox_inches="tight")
plt.show()

# FINAL TEST GATE

Run the next cell once, after the model identity and both thresholds are frozen. If either 95% target is missed, do not change the seed, split, feature set, weights, model, or thresholds after seeing the test results.

In [ ]:
assert TEST_SET_LOCKED and MODEL_AND_THRESHOLDS_FROZEN
y_test = _y_test_locked.copy()

if FINAL_MODEL_NAME == "OptimizedWeightedSoftVoting":
    test_matrix = np.column_stack([
        base_models[name].predict_proba(X_test_fit_locked)[:, 1]
        for name in base_names
    ])
    final_test_probabilities = test_matrix @ blend_weights
elif FINAL_MODEL_NAME == "OutOfFoldStacking":
    final_test_probabilities = stack_predict_proba(X_test_fit_locked)[:, 1]
else:
    raise RuntimeError(f"Unexpected frozen model: {FINAL_MODEL_NAME}")

operating_points = OrderedDict({
    "AccuracyMode": FINAL_ACCURACY_THRESHOLD,
    "HighPrecisionMode": FINAL_PRECISION_THRESHOLD,
})
final_rows = []
final_predictions = {}
for mode, threshold in operating_points.items():
    metrics = metric_bundle(y_test, final_test_probabilities, threshold)
    final_predictions[mode] = (final_test_probabilities >= threshold).astype(int)
    final_rows.append({
        "model": FINAL_MODEL_NAME,
        "operating_mode": mode,
        "threshold": threshold,
        **{key: metrics[key] for key in METRIC_COLUMNS},
        "tn": metrics["tn"], "fp": metrics["fp"],
        "fn": metrics["fn"], "tp": metrics["tp"],
        "accuracy_and_precision_both_at_least_95": bool(
            metrics["accuracy"] >= TARGET_ACCURACY
            and metrics["precision"] >= TARGET_PRECISION
        ),
    })

final_test_table = pd.DataFrame(final_rows)
final_test_table.to_csv(TABLE_DIR / "FINAL_UNTOUCHED_TEST_METRICS.csv", index=False)
display(final_test_table.style.format({c: "{:.4f}" for c in METRIC_COLUMNS + ["threshold"]}))

TARGET_ACHIEVED = bool(final_test_table["accuracy_and_precision_both_at_least_95"].any())
print("Accuracy and precision both >= 95% on untouched test:", TARGET_ACHIEVED)
if not TARGET_ACHIEVED:
    print("The 95% dual target was not achieved honestly. Report the actual values and do not retune on this test set.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (mode, threshold) in zip(axes, operating_points.items()):
    cm = confusion_matrix(y_test, final_predictions[mode], labels=[0, 1])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap=sns.light_palette("#6C8EBF", as_cmap=True),
        cbar=False,
        ax=ax,
        xticklabels=["Non-diabetic", "Diabetic"],
        yticklabels=["Non-diabetic", "Diabetic"],
    )
    ax.set(
        xlabel="Predicted",
        ylabel="Actual",
        title=f"{mode} (threshold={threshold:.3f})",
    )
fig.suptitle(f"Untouched-test confusion matrices: {FINAL_MODEL_NAME}")
plt.tight_layout()
plt.savefig(FIG_DIR / "final_test_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()

fpr, tpr, _ = roc_curve(y_test, final_test_probabilities)
test_auc = roc_auc_score(y_test, final_test_probabilities)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color="#6C8EBF", linewidth=2.8, label=f"AUC={test_auc:.3f}")
ax.plot([0, 1], [0, 1], "--", color="#777777")
ax.set(xlabel="False-positive rate", ylabel="True-positive rate", title="Final untouched-test ROC")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "final_test_roc.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Explainability of the proposed ensemble

Ensemble-level permutation importance and LIME are calculated on validation examples. Tree SHAP explains the CatBoost component because an optimized multi-model probability blend has no single native TreeSHAP representation. The test set is not used to revise features or explanations.

In [ ]:
def validation_final_probabilities(frame):
    if FINAL_MODEL_NAME == "OptimizedWeightedSoftVoting":
        matrix = np.column_stack([
            base_models[name].predict_proba(frame)[:, 1]
            for name in base_names
        ])
        return matrix @ blend_weights
    return stack_predict_proba(frame)[:, 1]

rng = np.random.default_rng(SEED)
baseline_auc = roc_auc_score(y_val, validation_final_probabilities(X_val_fit))
permutation_rows = []
for feature in EXPECTED_FEATURES:
    decreases = []
    for repeat in range(10):
        permuted = X_val_fit.copy()
        permuted[feature] = rng.permutation(permuted[feature].to_numpy())
        permuted_auc = roc_auc_score(y_val, validation_final_probabilities(permuted))
        decreases.append(baseline_auc - permuted_auc)
    permutation_rows.append({
        "feature": feature,
        "importance_mean_auc_decrease": float(np.mean(decreases)),
        "importance_std": float(np.std(decreases, ddof=1)),
    })
permutation_table = pd.DataFrame(permutation_rows).sort_values(
    "importance_mean_auc_decrease", ascending=False
)
permutation_table.to_csv(TABLE_DIR / "validation_ensemble_permutation_importance.csv", index=False)
display(permutation_table)

plot_data = permutation_table.sort_values("importance_mean_auc_decrease")
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(
    plot_data["feature"],
    plot_data["importance_mean_auc_decrease"],
    xerr=plot_data["importance_std"],
    color=SOFT[0],
)
ax.set(xlabel="Validation ROC-AUC decrease after permutation", title="Ensemble permutation importance")
plt.tight_layout()
plt.savefig(FIG_DIR / "validation_ensemble_permutation_importance.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import shap

shap_sample = X_val_fit.sample(n=min(1200, len(X_val_fit)), random_state=SEED)
shap_explainer = shap.TreeExplainer(base_models["CatBoost"])
shap_values = shap_explainer.shap_values(shap_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[-1]

shap.summary_plot(
    shap_values,
    shap_sample,
    feature_names=EXPECTED_FEATURES,
    plot_type="dot",
    show=False,
)
plt.title("Tree SHAP summary: CatBoost ensemble component")
plt.tight_layout()
plt.savefig(FIG_DIR / "validation_catboost_shap_summary.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_training = X_train_fit.sample(n=min(20_000, len(X_train_fit)), random_state=SEED)
lime_explainer = LimeTabularExplainer(
    lime_training.to_numpy(dtype=float),
    feature_names=EXPECTED_FEATURES,
    class_names=["Non-diabetic", "Diabetic"],
    mode="classification",
    discretize_continuous=True,
    random_state=SEED,
)

def lime_predict(values):
    frame = pd.DataFrame(values, columns=EXPECTED_FEATURES)
    positive = validation_final_probabilities(frame)
    return np.column_stack([1 - positive, positive])

validation_prob = validation_final_probabilities(X_val_fit)
case_index = int(np.argmin(np.abs(validation_prob - FINAL_ACCURACY_THRESHOLD)))
lime_explanation = lime_explainer.explain_instance(
    X_val_fit.iloc[case_index].to_numpy(dtype=float),
    lime_predict,
    num_features=len(EXPECTED_FEATURES),
)
lime_table = pd.DataFrame(lime_explanation.as_list(), columns=["condition", "weight"])
lime_table.to_csv(TABLE_DIR / "validation_ensemble_lime_explanation.csv", index=False)
display(lime_table)

fig = lime_explanation.as_pyplot_figure()
fig.suptitle(f"LIME validation explanation: case {case_index}")
fig.tight_layout()
fig.savefig(FIG_DIR / "validation_ensemble_lime_explanation.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Export results, models, and publication evidence

The downloadable ZIP includes the untouched-test metrics, prediction-level output, thresholds, ensemble weights, feature analyses, explanations, fitted models, figures, and environment versions.

In [ ]:
prediction_table = pd.DataFrame({
    "original_row_id": id_test,
    "actual_class": y_test.to_numpy(),
    "positive_probability": final_test_probabilities,
    "accuracy_mode_threshold": FINAL_ACCURACY_THRESHOLD,
    "accuracy_mode_prediction": final_predictions["AccuracyMode"],
    "high_precision_threshold": FINAL_PRECISION_THRESHOLD,
    "high_precision_prediction": final_predictions["HighPrecisionMode"],
})
prediction_table.to_csv(TABLE_DIR / "final_test_predictions.csv", index=False)

joblib.dump(
    {
        "base_models": base_models,
        "base_model_order": base_names,
        "weighted_voting_weights": blend_weights,
        "stacking_meta_model": stack_meta_model,
        "stacking_base_model_names": STACK_BASE_NAMES,
        "stacking_passthrough": True,
        "stacking_cv_folds": 3,
        "training_medians": train_medians,
        "final_model_name": FINAL_MODEL_NAME,
        "accuracy_threshold": FINAL_ACCURACY_THRESHOLD,
        "high_precision_threshold": FINAL_PRECISION_THRESHOLD,
        "feature_names": EXPECTED_FEATURES,
    },
    MODEL_DIR / "brfss_frozen_ensemble_bundle.joblib",
)

versions = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
try:
    import sklearn, xgboost, lightgbm, catboost
    versions.update({
        "scikit_learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
        "lightgbm": lightgbm.__version__,
        "catboost": catboost.__version__,
    })
except Exception:
    pass
(RESULTS_DIR / "environment_versions.json").write_text(json.dumps(versions, indent=2))

accuracy_row = final_test_table[final_test_table["operating_mode"] == "AccuracyMode"].iloc[0]
precision_row = final_test_table[final_test_table["operating_mode"] == "HighPrecisionMode"].iloc[0]
publication_note = f'''Dataset: original 253,680-row BRFSS 2015 binary health-indicator file.
Split: stratified 70% training / 15% validation / 15% untouched testing.
Final proposed model: {FINAL_MODEL_NAME}
Accuracy-mode test accuracy: {accuracy_row['accuracy']:.4f}
Accuracy-mode test precision: {accuracy_row['precision']:.4f}
High-precision-mode test precision: {precision_row['precision']:.4f}
High-precision-mode test recall: {precision_row['recall']:.4f}
Accuracy and precision both >=95% at one frozen operating point: {TARGET_ACHIEVED}

Report these actual values. Do not claim 95% if TARGET_ACHIEVED is False.
Do not reuse this test set for another round of model or threshold selection.
Recall and sensitivity are numerically identical for the positive class.
'''
(RESULTS_DIR / "publication_reporting_note.txt").write_text(publication_note)
print(publication_note)

archive_path = shutil.make_archive(str(RESULTS_DIR), "zip", root_dir=RESULTS_DIR)
print("Created:", archive_path)
try:
    from google.colab import files
    files.download(archive_path)
except Exception:
    pass

## 10. Publication rules

- Use the actual untouched-test values; a target is not a result.
- Report positive-class precision together with recall, confusion counts, ROC-AUC, MCC, specificity, sensitivity, and log loss.
- State that SMOTE was not used in the primary high-accuracy/high-precision analysis. If a SMOTE sensitivity analysis is later added, apply it inside training folds only.
- Do not compare this imbalanced BRFSS result directly with a balanced 50/50 dataset or a small clinical/symptom dataset without explaining the different prevalence and predictors.
- A publishable contribution should emphasize the leakage-safe ensemble, threshold trade-off, explainability, and external-validation need—not only a larger accuracy number.